# 실습 2: 협업 필터링 Explained (한 줄씩 분해하기)

이 실습은 `lab_01`에서 실행했던 User-based CF 파이프라인을 **한 줄씩 분해**하여, 각 코드가 어떤 원리로 동작하는지 이해하는 것을 목표로 합니다. 코사인 유사도 복습부터 시작해 CF 코드 전체를 해부합니다.

**개념 복기 및 이론 점검**
- 코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지를 재는 수치입니다. 평점 벡터에서 '방향이 비슷하다'는 것은 무엇을 의미하는가?
- `torch.mm(normalized, normalized.T)`가 왜 모든 사용자 쌍의 코사인 유사도를 한 번에 계산하는가?
- User-based CF와 Item-based CF 중 대규모 서비스에서 실제로 더 많이 쓰이는 방식은 무엇이고 왜인가?
- Sparsity(희소성)와 Cold Start(콜드 스타트)는 각각 어떤 상황에서 발생하는 문제인가?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/4주차/lab_02_explain.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

## 0. 임베딩과 코사인 유사도 복습

협업 필터링의 핵심 도구인 코사인 유사도를 작은 예시로 먼저 이해합니다.

**임베딩(Embedding)** 이란 어떤 대상(사용자, 아이템, 단어 등)을 숫자 벡터로 표현하는 것입니다. 협업 필터링에서 각 사용자의 평점 벡터 자체가 곧 그 사용자의 임베딩입니다.

**코사인 유사도(Cosine Similarity)** 는 두 벡터가 같은 방향을 얼마나 가리키는지를 -1 ~ 1 사이의 값으로 나타냅니다. 1이면 완전히 같은 방향, 0이면 직각, -1이면 반대 방향입니다.

In [ ]:
import torch
import torch.nn.functional as F

# 사용자 A, B, C의 영화 평점 벡터 (영화 5편에 대한 평점)
user_a = torch.tensor([5.0, 4.0, 0.0, 0.0, 1.0])  # 액션 좋아함
user_b = torch.tensor([4.0, 5.0, 0.0, 0.0, 2.0])  # A와 비슷한 취향
user_c = torch.tensor([0.0, 0.0, 5.0, 4.0, 0.0])  # A와 다른 취향 (로맨스)

sim_ab = F.cosine_similarity(user_a.unsqueeze(0), user_b.unsqueeze(0)).item()
sim_ac = F.cosine_similarity(user_a.unsqueeze(0), user_c.unsqueeze(0)).item()

print(f"A와 B의 코사인 유사도: {sim_ab:.4f}  → 취향이 비슷함")
print(f"A와 C의 코사인 유사도: {sim_ac:.4f}  → 취향이 다름")

### 🔬 코드 해설
- **`F.cosine_similarity(a, b)`**: `(a · b) / (‖a‖ × ‖b‖)` 를 계산합니다. 내적을 두 벡터의 크기로 나누어 방향만 비교합니다.
    - 평점 벡터에서 '방향이 같다'는 것은 **어떤 장르에 높은 점수를 주고 어떤 장르에 낮은 점수를 주는 패턴이 유사하다**는 의미입니다. 실제 점수의 크기보다 패턴이 중요합니다.
- **`.unsqueeze(0)`**: 1D 벡터 `(5,)` → 2D 행렬 `(1, 5)` 로 차원을 추가합니다. `F.cosine_similarity`가 배치 차원을 필요로 하기 때문입니다.
- A, B 두 사람은 같은 영화에 높은 점수를 주는 패턴이 비슷해서 유사도가 높고, A와 C는 서로 다른 영화에 집중해서 유사도가 낮게 나옵니다.

## 1-1. 환경 준비: 라이브러리 임포트

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

### 🔬 코드 해설
- `pandas`: 테이블 형태의 데이터를 다루는 라이브러리입니다. CSV 파일 로딩과 데이터 전처리에 사용합니다.
- `torch`: 텐서 연산 및 코사인 유사도 계산에 사용합니다. NumPy보다 GPU 연산 지원과 행렬 연산이 편리합니다.
- `torch.nn.functional as F`: `F.cosine_similarity` 등 함수형 API를 사용합니다.
- `torch.manual_seed(42)`: 난수 시드를 고정해 실험의 재현성을 보장합니다.

## 1-2. 데이터 다운로드

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O ml-100k.zip
!unzip -q -o ml-100k.zip
print("다운로드 완료")

### 🔬 코드 해설
- **MovieLens ml-100k**: GroupLens 연구소에서 수집한 공개 영화 평점 데이터셋입니다. 943명의 사용자가 1682편의 영화에 매긴 100,000개의 평점을 담고 있습니다.
- **`u.data`**: 탭(\t)으로 구분된 `user_id | item_id | rating | timestamp` 형식의 파일입니다.
- **`u.item`**: 영화 ID와 영화 제목 등 메타데이터를 담은 파일입니다. 추천 결과를 사람이 읽을 수 있는 형태로 출력할 때 사용합니다.

## 2. User-Item 평점 행렬 구성

In [ ]:
df = pd.read_csv(
    'ml-100k/u.data', sep='\t', header=None,
    names=['user_id', 'item_id', 'rating', 'timestamp']
)

movies = pd.read_csv(
    'ml-100k/u.item', sep='|', header=None, encoding='latin-1',
    usecols=[0, 1], names=['item_id', 'title']
)
movie_names = dict(zip(movies['item_id'], movies['title']))

n_users = df['user_id'].max()
n_items = df['item_id'].max()

ratings = torch.zeros(n_users, n_items)
for row in df.itertuples():
    ratings[row.user_id - 1, row.item_id - 1] = row.rating

print(f"행렬 크기: {ratings.shape}  (사용자 × 영화)")
print(f"평균 평점: {df['rating'].mean():.2f} / 5.0")

### 🔬 코드 해설
- **`pd.read_csv(...)`**: 평점 데이터 파일을 불러옵니다. `sep='\t'`는 탭 구분자, `header=None`은 첫 줄에 헤더가 없음을 의미합니다.
- **`torch.zeros(n_users, n_items)`**: 943 × 1682 크기의 0으로 채워진 행렬을 만듭니다. 0은 '아직 평가하지 않음'을 의미합니다.
- **`ratings[row.user_id - 1, row.item_id - 1] = row.rating`**: ml-100k의 ID는 1부터 시작하므로 -1을 빼서 0-indexed 텐서 인덱스로 맞춥니다.
- 결과 행렬의 각 **행**은 한 사용자의 평점 벡터, 각 **열**은 한 영화에 대한 모든 사용자의 평점 벡터입니다. 대부분의 칸이 0인 이 구조를 **희소 행렬(Sparse Matrix)** 이라고 합니다.

## 3. 코사인 유사도 계산 — 핵심 원리

모든 사용자 쌍(943 × 943)의 코사인 유사도를 한 번에 계산합니다.

In [ ]:
# Step 1: 각 사용자의 평점 벡터를 단위 벡터로 정규화
norms = torch.norm(ratings, dim=1, keepdim=True)  # (943, 1) — 각 사용자 벡터의 크기
norms = norms.clamp(min=1e-8)                     # 0으로 나누는 것 방지
normalized = ratings / norms                       # (943, 1682) — 모든 벡터를 길이 1로 만듦

# Step 2: 행렬 곱으로 모든 쌍의 코사인 유사도를 한 번에 계산
user_sim = torch.mm(normalized, normalized.T)     # (943, 943)

print(f"유사도 행렬 크기: {user_sim.shape}")
print(f"대각선(자기 자신과의 유사도): {user_sim.diagonal()[:5]}")
print(f"사용자 0과 유사도가 가장 높은 상위 5명:")
top5 = torch.topk(user_sim[0], 6).indices[1:]  # 자기 자신(0번) 제외
for uid in top5:
    print(f"  사용자 {uid.item() + 1}번: 유사도 {user_sim[0, uid].item():.4f}")

### 🔬 코드 해설

#### 왜 정규화 후 행렬 곱인가?

코사인 유사도 공식은 `cos(a, b) = (a · b) / (‖a‖ × ‖b‖)` 입니다.

- **Step 1**: 분모를 미리 처리합니다. 각 벡터를 자신의 크기(`norm`)로 나누면, 방향만 남고 크기가 1인 **단위 벡터**가 됩니다. 이를 정규화(normalization)라고 합니다.
- **Step 2**: 단위 벡터끼리의 내적(dot product)은 곧 코사인 유사도입니다. 행렬 `A`와 `A^T`의 곱 `A @ A^T`는 `A`의 모든 행 쌍의 내적을 한 번에 계산합니다. 즉, `result[i][j]`가 사용자 i와 사용자 j의 코사인 유사도가 됩니다.

- **`dim=1`**: `torch.norm`에서 행 방향(각 사용자별)으로 크기를 계산합니다.
- **`.clamp(min=1e-8)`**: 한 번도 평점을 매기지 않은 사용자(모든 값이 0)의 벡터 크기가 0이 되어 나눗셈 오류가 나는 것을 방지합니다. 아주 작은 값(1e-8)으로 대체합니다.
- **`torch.mm` vs `F.cosine_similarity`**: `F.cosine_similarity`는 두 벡터 쌍을 비교할 때 편리하지만, 943×943 모든 쌍을 구하려면 반복문이 필요합니다. `torch.mm`(행렬 곱)은 이를 **한 번의 연산**으로 처리하므로 훨씬 빠릅니다.

## 4. User-based CF: 이웃 선정

In [ ]:
user_idx = 0
top_n_neighbors = 20

sim_scores = user_sim[user_idx].clone()  # 사용자 0번의 유사도 벡터 복사
sim_scores[user_idx] = -1               # 자기 자신은 추천에서 제외

top_neighbors = torch.topk(sim_scores, top_n_neighbors).indices

print(f"사용자 {user_idx + 1}번의 이웃 사용자 ID (상위 {top_n_neighbors}명):")
print([uid.item() + 1 for uid in top_neighbors])
print(f"\n이웃들의 유사도 값:")
print([f"{v:.3f}" for v in sim_scores[top_neighbors].tolist()])

### 🔬 코드 해설
- **`.clone()`**: 원본 `user_sim` 행렬을 수정하지 않기 위해 복사본을 만듭니다. Python에서 텐서 슬라이싱은 뷰(view)를 반환하므로, 수정 시 원본도 바뀔 수 있습니다.
- **`sim_scores[user_idx] = -1`**: 자기 자신과의 유사도는 항상 1이므로(가장 높음), -1로 설정해 추천 대상에서 명시적으로 제외합니다.
- **`torch.topk(sim_scores, k)`**: 유사도 값이 가장 높은 k명의 값과 인덱스를 반환합니다. 이 k명이 바로 이웃(neighbor)입니다. 이웃 수(k)는 하이퍼파라미터로, 너무 적으면 추천이 편향되고 너무 많으면 유사도가 낮은 사용자까지 포함됩니다.

## 5. 추천 생성 — 유사도 가중 합산

In [ ]:
top_n_items = 10

unrated_mask = (ratings[user_idx] == 0)          # 아직 보지 않은 영화 위치

neighbor_ratings = ratings[top_neighbors]         # (20, 1682) — 이웃들의 평점 행렬
weights = sim_scores[top_neighbors].unsqueeze(1)  # (20, 1)  — 각 이웃의 유사도

scores = (neighbor_ratings * weights).sum(dim=0)  # (1682,) — 유사도 가중 합산
scores[~unrated_mask] = -float('inf')             # 이미 평가한 영화 제외

top_items = torch.topk(scores, top_n_items).indices

print(f"사용자 {user_idx + 1}번의 추천 영화 Top {top_n_items}:")
for rank, item_idx in enumerate(top_items.tolist(), 1):
    print(f"  {rank}. {movie_names.get(item_idx + 1, 'Unknown')}  (점수: {scores[item_idx].item():.2f})")

### 🔬 코드 해설
- **`unrated_mask`**: 타깃 사용자가 아직 평가하지 않은 영화(`rating == 0`)만 추천 대상으로 삼습니다. 이미 본 영화를 다시 추천하는 것은 의미 없습니다.
- **`neighbor_ratings * weights`**: 유사도가 높은 이웃의 평점에 더 높은 가중치를 줍니다. 예를 들어 유사도 0.9인 이웃이 5점을 준 영화는, 유사도 0.1인 이웃이 5점을 준 영화보다 더 높은 점수를 받습니다.
- **`.sum(dim=0)`**: 모든 이웃의 가중 평점을 영화별로 합산합니다. `dim=0`은 행 방향으로 합산한다는 의미입니다.
- **`scores[~unrated_mask] = -float('inf')`**: 이미 본 영화의 점수를 음의 무한대로 설정해, `topk` 결과에 절대 등장하지 않도록 합니다.

> 💡 **직접 해보기**: `top_n_neighbors`를 5, 50으로 바꾸면 추천 결과가 어떻게 달라지나요? `user_idx`를 바꾸면 어떤 영화가 추천되나요?

## 6. Item-based CF와의 비교

User-based CF가 '유사한 사람들이 좋아한 것'을 추천한다면, **Item-based CF**는 '내가 좋아했던 것과 유사한 것'을 추천합니다.

| | User-based CF | Item-based CF |
|---|---|---|
| **유사도 계산 대상** | 사용자 간 | 아이템 간 |
| **추천 아이디어** | '나와 비슷한 사람들이 좋아한 것' | '내가 좋아했던 것과 비슷한 것' |
| **행렬 크기** | 사용자 수 × 사용자 수 | 아이템 수 × 아이템 수 |
| **업데이트 빈도** | 새 사용자 추가 시마다 재계산 | 아이템이 상대적으로 안정적 → 사전 계산 가능 |
| **실제 사례** | (소규모) | 아마존 "함께 구매한 상품", 대규모 서비스 |

아마존과 같은 대규모 서비스에서는 아이템 수보다 사용자 수가 훨씬 많이 변하므로, 아이템 유사도를 사전에 계산해두는 Item-based CF가 실시간 추천 속도가 빠릅니다.

## 7. 한계와 다음 단계 — 메모리 기반 CF의 두 가지 벽

지금까지 구현한 방식(메모리 기반 CF)은 직관적이지만, 현실 서비스에서 두 가지 구조적 한계에 부딪힙니다.

### 희소성 (Sparsity)
실제 평점 행렬은 대부분의 칸이 비어 있습니다. ml-100k의 경우 약 93.7%가 0입니다. 사용자가 많고 아이템이 많을수록 이 문제는 심해집니다. 공통으로 평가한 아이템이 거의 없는 두 사용자 사이의 코사인 유사도는 신뢰하기 어렵습니다.

### 콜드 스타트 (Cold Start)
신규 사용자는 평점 이력이 없으므로 평점 벡터가 전부 0입니다. 코사인 유사도 계산 자체가 불가능해 추천을 만들 수 없습니다. 신규 아이템도 마찬가지입니다.

### Matrix Factorization(행렬 분해)
이 두 한계를 극복하기 위해 등장한 방법이 **Matrix Factorization(MF)** 입니다. 평점 행렬 R을 두 개의 저차원 행렬(사용자 잠재 벡터 U, 아이템 잠재 벡터 V)로 분해해 빈칸을 채우는 방식입니다.

```
R ≈ U · V^T
```

U와 V는 직접 관찰할 수 없는 **잠재 요인(Latent Factor)** 을 담습니다. 예를 들어 사용자의 장르 선호도나 영화의 장르 특성 같은 것들입니다. 이 표현이 3주차에서 배운 **임베딩** 개념과 직접 연결됩니다. 2009년 Netflix Prize에서 이 방식이 기존 CF를 압도하면서 널리 알려졌습니다.

## 8. ✅ 학습 결과 정리
- 코사인 유사도가 평점 벡터의 '방향 유사성'을 측정하며, 정규화 후 행렬 곱으로 효율적으로 계산되는 원리를 이해했습니다.
- User-based CF의 세 단계(유사도 계산 → 이웃 선정 → 가중 합산)를 코드 한 줄씩 분해했습니다.
- Item-based CF와의 구조적 차이, 그리고 각각이 유리한 상황을 비교했습니다.
- 메모리 기반 CF의 두 한계(Sparsity, Cold Start)와 Matrix Factorization의 등장 맥락을 파악했습니다.
- 🎯 **핵심 결론:** 다음 실습(`lab_03`)에서는 데이터 탐색(EDA)으로 희소성을 직접 눈으로 확인하고, Item-based CF를 직접 구현해 봅니다.